In [3]:
import torch
from mmasim_kernels.amd.cdna3 import mfma_kernels
suffix = "f32_16x16x32_fp8_fp8"
kernel = mfma_kernels[suffix]

In [23]:
K = 32
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-9, 2.**-9
a = torch.full([K], va, dtype=torch.float8_e4m3fnuz)
b = torch.full([K], vb, dtype=torch.float8_e4m3fnuz)
c = torch.tensor(va*vb, dtype=torch.float32)
# aibi = U, ajbj = -U
for i in range(K):
    a[i], b[i] = Ua, Ub
    print(i, end=':\t')
    for j in range(i):
        a[j], b[j] = Ua, -Ub
        print(int(kernel.dpa(a, b, c).item() / (va*vb)), end=' ')
        a[j], b[j] = va, vb
    print()
    a[i], b[i] = va, vb
# c = U, ajbj = -U
c[None] = Ua*Ub
print("c", end=':\t')
for j in range(K):
    a[j], b[j] = Ua, -Ub
    print(int(kernel.dpa(a, b, c).item() / (va*vb)), end=' ')
    a[j], b[j] = va, vb
print()
c[None] = va*vb


0:	
1:	16 
2:	16 16 
3:	16 16 16 
4:	16 16 16 16 
5:	16 16 16 16 16 
6:	16 16 16 16 16 16 
7:	16 16 16 16 16 16 16 
8:	16 16 16 16 16 16 16 16 
9:	16 16 16 16 16 16 16 16 16 
10:	16 16 16 16 16 16 16 16 16 16 
11:	16 16 16 16 16 16 16 16 16 16 16 
12:	16 16 16 16 16 16 16 16 16 16 16 16 
13:	16 16 16 16 16 16 16 16 16 16 16 16 16 
14:	16 16 16 16 16 16 16 16 16 16 16 16 16 16 
15:	16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
17:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
18:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
19:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
20:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
21:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
22:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
23:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
24:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
25:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
26:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
27:	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [ ]:
# minimal precision of c
K = 16
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
Ua, Ub = 2.**7, 2.**7
# aibi = U, ajbj = -U, c = 2**-e
for i in range(K):
    print(i, end=':\t')
    a[i], b[i] = Ua, Ub
    for j in range(i):
        a[j], b[j] = Ua, -Ub
        for e in range(30):
            c[None] = Ua*Ub * 2.**-e
            if kernel.dpa(a, b, c).item() != Ua * Ub * 2.**-e:
                print(e, end=' ')
                break
        a[j], b[j] = 0, 0
    print()
    a[i], b[i] = 0, 0



0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
8:	25 25 25 25 25 25 25 25 
9:	25 25 25 25 25 25 25 25 25 
10:	25 25 25 25 25 25 25 25 25 25 
11:	25 25 25 25 25 25 25 25 25 25 25 
12:	25 25 25 25 25 25 25 25 25 25 25 25 
13:	25 25 25 25 25 25 25 

25 25 25 25 25 25 
14:	25 25 25 25 25 25 25 25 25 25 25 25 25 25 
15:	25 25 25 25 25 25 25 25 25 25 25 25 25 25 25 


In [ ]:
# minimal precision of akbk
# akbk = 2**-e, aibi = U, ajbj = -U
Ua, Ub = 2.**7, 2.**7
for k in range(K):
    a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
    b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
    c = torch.tensor(0., dtype=torch.float32)
    print("k =", k)
    for i in range(k):
        print(i, end=':\t')
        a[i], b[i] = Ua, Ub
        for j in range(i):
            a[j], b[j] = Ua, -Ub
            for e in range(30):
                a[k], b[k] = Ua * 2.**-(e//2), Ub * 2.**-(e//2)
                if e%2 != 0:
                    a[k] = a[k].item() * 0.5
                if kernel.dpa(a, b, c).item() != Ua * Ub * 2.**-e:
                    print(e, end=' ')
                    break
            a[j], b[j] = 0, 0
        print()
        a[i], b[i] = 0, 0
    # akbk = 2**-e, c = U, ajbj = -U
    print("c", end=':\t')
    c[None] = Ua * Ub
    for j in range(k):
        a[j], b[j] = Ua, -Ub
        for e in range(30):
            a[k], b[k] = Ua * 2.**-(e//2), Ub * 2.**-(e//2)
            if e%2 != 0:
                a[k] = a[k].item() * 0.5
            if kernel.dpa(a, b, c).item() != Ua * Ub * 2.**-e:
                print(e, end=' ')
                break
        a[j], b[j] = 0, 0
    print()
    c[None] = 0.0




k = 0
c:	
k = 1
0:	
c:	25 
k = 2
0:	
1:	25 
c:	25 25 
k = 3
0:	
1:	25 
2:	25 25 
c:	25 25 25 
k = 4
0:	
1:	25 
2:	25 25 
3:	25 25 25 
c:	25 25 25 25 
k = 5
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
c:	25 25 25 25 25 
k = 6
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
c:	25 25 25 25 25 25 
k = 7
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
c:	25 25 25 25 

25 25 25 
k = 8
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
c:	25 25 25 25 25 25 25 25 
k = 9
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
8:	25 25 25 25 25 25 25 25 
c:	25 25 25 25 25 25 25 25 25 
k = 10
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
8:	25 25 25 25 25 25 25 25 
9:	25 25 25 25 25 25 25 25 25 
c:	25 25 25 25 25 25 25 25 25 25 
k = 11
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
8:	25 25 25 25 25 25 25 25 
9:	25 25 25 25 25 25 25 25 25 
10:	25 25 25 25 25 25 25 25 25 25 
c:	25 25 25 25 25 25 25 25 25 25 25 
k = 12
0:	
1:	25 
2:	25 25 
3:	25 25 25 
4:	25 25 25 25 
5:	25 25 25 25 25 
6:	25 25 25 25 25 25 
7:	25 25 25 25 25 25 25 
8:	25 25 25 25 25 25 25 25 
9:	25 25 25 25 25 25 25 25 25 
10:	25 

In [ ]:
# internal summation order before exp-decrease normalization
K = 16
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-6, 2.**-5
a = torch.full([K], va, dtype=torch.float8_e4m3fnuz)
b = torch.full([K], vb, dtype=torch.float8_e4m3fnuz)
c = torch.tensor(vb*vb, dtype=torch.float32)
# aibi = U, ajbj = -U
for i in range(K):
    a[i], b[i] = Ua, Ub
    print(i, end=':\t')
    for j in range(i):
        a[j], b[j] = Ua, -Ub
        print(int(kernel.dpa(a, b, c).item() / (vb*vb)), end=' ')
        a[j], b[j] = va, vb
    print()
    a[i], b[i] = va, vb
# c = U, ajbj = -U
c[None] = Ua*Ub
print("c", end=':\t')
for j in range(K):
    a[j], b[j] = Ua, -Ub
    print(int(kernel.dpa(a, b, c).item() / (va*vb)), end=' ')
    a[j], b[j] = va, vb
print()
c[None] = va*vb


0:	
1:	1 
2:	5 1 
3:	1 5 1 
4:	5 1 5 1 
5:	1 5 1 5 1 
6:	5 1 5 1 5 1 
7:	1 5 1 5 1 5 1 
8:	5 1 5 1 5 1 5 1 
9:	1 5 1 5 1 5 1 5 1 
10:	5 1 5 1 5 1 5 1 5 1 
11:	1 5 1 5 1 5 1 5 1 5 1 
12:	5 1 5 1 5 1 5 1 5 1 5 1 
13:	1 5 1 5 1 5 1 5 1 5 1 5 1 
14:	5 1 5 1 5 1 5 1 5 1 5 1 5 1 
15:	1 5 1 5 1 5 1 5 1 5 1 5 1 5 1 
c:	8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 


In [ ]:
# first-level fused sum: rounding mode
# precision tested
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-5
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = Ua, Ub
a[2], b[2] = Ua, -Ub
for s in [0.25, 0.75, -0.25, -0.75, 0.5, 1.5, -0.5, -1.5]:
    a[4], b[4] = s * va, vb
    print(s, kernel.dpa(a, b, c).item().hex())

0.25 0x0.0p+0
0.75 0x0.0p+0
-0.25 0x0.0p+0
-0.75 0x0.0p+0
0.5 0x0.0p+0
1.5 0x1.0000000000000p-10
-0.5 0x0.0p+0
-1.5 -0x1.0000000000000p-10


In [44]:
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-5
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[1], b[1] = Ua, Ub
a[3], b[3] = Ua, -Ub
for s in [0.25, 0.75, -0.25, -0.75, 0.5, 1.5, -0.5, -1.5]:
    a[5], b[5] = s * va, vb
    print(s, kernel.dpa(a, b, c).item().hex())

0.25 0x0.0p+0
0.75 0x0.0p+0
-0.25 0x0.0p+0
-0.75 0x0.0p+0
0.5 0x0.0p+0
1.5 0x1.0000000000000p-10
-0.5 0x0.0p+0
-1.5 -0x1.0000000000000p-10


In [72]:
# first-level fused sum: s>=2 -> exp increase ?
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = 1.5 * Ua, 1.5 * Ub
a[2], b[2] = 1.5 * Ua, -1.5 * Ub
for e in range(20, 30):
    a[4], b[4] = Ua * 2.**-(e//2), Ub * 2.**-(e//2)
    if e%2 != 0:
        a[4] = a[4].item() * 0.5
    print(e, kernel.dpa(a, b, c).item().hex())

20 0x1.0000000000000p-6
21 0x1.0000000000000p-7
22 0x1.0000000000000p-8
23 0x1.0000000000000p-9
24 0x1.0000000000000p-10
25 0x0.0p+0
26 0x0.0p+0
27 0x0.0p+0
28 0x0.0p+0
29 0x0.0p+0


In [ ]:
# second-level fused sum: rounding mode
# precision tested
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-5
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = Ua, Ub
a[2], b[2] = Ua, -Ub
for s in [0.25, 0.75, -0.25, -0.75, 0.5, 1.5, -0.5, -1.5]:
    a[1], b[1] = s * va, vb
    print(s, kernel.dpa(a, b, c).item().hex())

0.25 0x0.0p+0
0.75 0x0.0p+0
-0.25 -0x1.0000000000000p-10
-0.75 -0x1.0000000000000p-10
0.5 0x0.0p+0
1.5 0x1.0000000000000p-10
-0.5 -0x1.0000000000000p-10
-1.5 -0x1.0000000000000p-9


In [46]:
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-5
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[1], b[1] = Ua, Ub
a[3], b[3] = Ua, -Ub
for s in [0.25, 0.75, -0.25, -0.75, 0.5, 1.5, -0.5, -1.5]:
    a[0], b[0] = s * va, vb
    print(s, kernel.dpa(a, b, c).item().hex())

0.25 0x0.0p+0
0.75 0x0.0p+0
-0.25 -0x1.0000000000000p-10
-0.75 -0x1.0000000000000p-10
0.5 0x0.0p+0
1.5 0x1.0000000000000p-10
-0.5 -0x1.0000000000000p-10
-1.5 -0x1.0000000000000p-9


In [73]:
# second-level fused sum: s>=2 -> exp increase ?
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = Ua, Ub
a[2], b[2] = Ua, Ub
c[None] = -2 * Ua * Ub
for e in range(20, 30):
    a[1], b[1] = Ua * 2.**-(e//2), Ub * 2.**-(e//2)
    if e%2 != 0:
        a[1] = a[1].item() * 0.5
    # a[3] = a[1]
    # b[3] = b[1]
    print(e, kernel.dpa(a, b, c).item().hex())

20 0x1.0000000000000p-6
21 0x1.0000000000000p-7
22 0x1.0000000000000p-8
23 0x1.0000000000000p-9
24 0x1.0000000000000p-10
25 0x0.0p+0
26 0x0.0p+0
27 0x0.0p+0
28 0x0.0p+0
29 0x0.0p+0


In [ ]:
# last-level fused sum: c rounding mode
# c precision tested
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-5
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = Ua, Ub
a[2], b[2] = Ua, -Ub
for s in [0.25, 0.75, -0.25, -0.75, 0.5, 1.5, -0.5, -1.5]:
    c[None] = s * va * vb
    print(s, kernel.dpa(a, b, c).item().hex())
for s in [-1.0, -0.5, -0.25, -0.125, -0.0625]:
    c[None] = s * va * vb
    print(s, kernel.dpa(a, b, c).item().hex())


0.25 0x0.0p+0
0.75 0x0.0p+0
-0.25 0x0.0p+0
-0.75 -0x1.0000000000000p-10
0.5 0x0.0p+0
1.5 0x1.0000000000000p-10
-0.5 -0x1.0000000000000p-10
-1.5 -0x1.0000000000000p-9
-1.0 -0x1.0000000000000p-10
-0.5 -0x1.0000000000000p-10
-0.25 0x0.0p+0
-0.125 0x0.0p+0
-0.0625 0x0.0p+0


In [ ]:
# last-level fused sum: s>=2 -> exp increase ?
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = Ua, Ub
for e in range(20, 30):
    c[None] = -Ua * Ub * 2.**-e
    print(e, kernel.dpa(a, b, c).item().hex())
a[0], b[0] = .5*Ua, Ub
a[1], b[1] = .5*Ua, Ub
for e in range(20, 30):
    c[None] = -Ua * Ub * 2.**-e
    print(e, kernel.dpa(a, b, c).item().hex())


20 0x1.ffffe00000000p+13
21 0x1.fffff00000000p+13
22 0x1.fffff80000000p+13
23 0x1.fffffc0000000p+13
24 0x1.fffffe0000000p+13
25 0x1.fffffe0000000p+13
26 0x1.0000000000000p+14
27 0x1.0000000000000p+14
28 0x1.0000000000000p+14
29 0x1.0000000000000p+14
20 0x1.ffffe00000000p+13
21 0x1.fffff00000000p+13
22 0x1.fffff80000000p+13
23 0x1.fffffc0000000p+13
24 0x1.fffffe0000000p+13
25 0x1.0000000000000p+14
26 0x1.0000000000000p+14
27 0x1.0000000000000p+14
28 0x1.0000000000000p+14
29 0x1.0000000000000p+14


In [62]:
# last-level fused sum: output rounding mode
Ua, Ub = 2.**7, 2.**7
va, vb = 2.**-5, 2.**-4
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
a[0], b[0] = 0.25 * Ua, Ub
a[2], b[2] = 0.25 * Ua, Ub
a[4], b[4] = 0.25 * Ua, Ub
a[6], b[6] = 0.25 * Ua, Ub
for s in [0.25, 0.75, 0.5, 1.5]:
    c[None] = s * va * vb
    print(s, kernel.dpa(a, b, c).item().hex())
    print(-s, kernel.dpa((-a.float()).to(b.dtype), b, -c).item().hex())

0.25 0x1.0000000000000p+14
-0.25 -0x1.0000000000000p+14
0.75 0x1.0000020000000p+14
-0.75 -0x1.0000020000000p+14
0.5 0x1.0000000000000p+14
-0.5 -0x1.0000000000000p+14
1.5 0x1.0000040000000p+14
-1.5 -0x1.0000040000000p+14


In [66]:
# last-level fused sum: non-c precision
a = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
b = torch.zeros([K], dtype=torch.float8_e4m3fnuz)
c = torch.tensor(0., dtype=torch.float32)
c[None] = 2.**24
a[0], b[0] = 1., 1.
for e in range(10):
    a[2], b[2] = 2.**-(e//2), 2.**-(e//2)
    if e%2 != 0:
        a[2] = a[2].item() * 0.5
    print(e, kernel.dpa(a, b, c).item().hex())


0 0x1.0000020000000p+24
1 0x1.0000020000000p+24
2 0x1.0000020000000p+24
3 0x1.0000020000000p+24
4 0x1.0000020000000p+24
5 0x1.0000020000000p+24
6 0x1.0000020000000p+24
7 0x1.0000020000000p+24
8 0x1.0000000000000p+24
9 0x1.0000000000000p+24


In [ ]:
for i in range(10):
    ta = torch.tensor([1.0, 2.**-i], dtype=torch.float8_e4m3fnuz, device='cuda:0')
    tb = torch.tensor([1., 1.], dtype=torch.float8_e4m3fnuz, device='cuda:0')
    tc = torch.tensor(float.fromhex("0x1.000008p24"), device='cuda:0', dtype=torch.float32)
    print(i, kernel.dpa(ta, tb, tc).item().hex())


0 0x1.00000a0000000p+24
1 0x1.00000a0000000p+24
2 0x1.00000a0000000p+24
3 0x1.00000a0000000p+24
4 0x1.00000a0000000p+24
5 0x1.00000a0000000p+24
6 0x1.00000a0000000p+24
7 0x1.00000a0000000p+24
8 0x1.0000080000000p+24
9 0x1.0000080000000p+24


In [ ]:
for i in range(10):
    ta = torch.tensor([8.+2., 2.**-i], dtype=torch.float8_e4m3fnuz, device='cuda:0')
    tb = torch.tensor([1., 1.], dtype=torch.float8_e4m3fnuz, device='cuda:0')
    tc = torch.tensor(float.fromhex("0x1.fffff8p24"), device='cuda:0', dtype=torch.float32)
    print(i, kernel.dpa(ta, tb, tc).item().hex())


0 0x1.0000020000000p+25
1 0x1.0000020000000p+25
2 0x1.0000020000000p+25
3 0x1.0000020000000p+25
4 0x1.0000020000000p+25
5 0x1.0000020000000p+25
6 0x1.0000020000000p+25
7 0x1.0000000000000p+25
8 0x1.0000000000000p+25
9 0x1.0000000000000p+25
